In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# Enzyme / Non-Enzyme classification using Multi-Property Pseudo-AAC + sequence length + CTD + windowed AAC + XGBoost
import os, re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from joblib import Parallel, delayed
import multiprocessing
import xgboost as xgb
import joblib

# ────────────────────────────────────────────────────────────
# 0. Pseudo-AAC parameters
LAG = 10
W   = 0.05
AA20 = "ACDEFGHIKLMNPQRSTVWY"

# 1. FASTA reader
def read_fasta(fasta_path):
    seqs, seq = [], ""
    with open(fasta_path, "r") as f:
        for line in f:
            if line.startswith(">"):
                if seq:
                    seqs.append(seq)
                    seq = ""
            else:
                seq += line.strip()
        if seq:
            seqs.append(seq)
    return seqs

# ────────────────────────────────────────────────────────────
# 2. Physicochemical property tables (dict)
HYDRO = {
    'A':1.8,'C':2.5,'D':-3.5,'E':-3.5,'F':2.8,
    'G':-0.4,'H':-3.2,'I':4.5,'K':-3.9,'L':3.8,
    'M':1.9,'N':-3.5,'P':-1.6,'Q':-3.5,'R':-4.5,
    'S':-0.8,'T':-0.7,'V':4.2,'W':-0.9,'Y':-1.3
}
POLAR = {
    'A':8.1,'C':5.5,'D':13.0,'E':12.3,'F':5.2,
    'G':9.0,'H':10.4,'I':5.2,'K':11.3,'L':4.9,
    'M':5.7,'N':11.6,'P':8.0,'Q':10.5,'R':10.5,
    'S':9.2,'T':8.6,'V':5.9,'W':5.4,'Y':6.2
}
CHARGE = {
    'A':0,'C':0,'D':-1,'E':-1,'F':0,
    'G':0,'H':0.5,'I':0,'K':1,'L':0,
    'M':0,'N':0,'P':0,'Q':0,'R':1,
    'S':0,'T':0,'V':0,'W':0,'Y':0
}
PROPERTIES = {"hydro": HYDRO, "polar": POLAR, "charge": CHARGE}

# ────────────────────────────────────────────────────────────
# 3. Multi-Property Pseudo-AAC + sequence length
def pse_aac_multi(seq, lam=LAG, w=W, props=PROPERTIES, add_len=True):
    seq = re.sub("[^ACDEFGHIKLMNPQRSTVWY]", "", seq.upper())
    L = len(seq)
    if L == 0:
        base_feat = np.zeros(20 + lam * len(props), dtype=float)
        return np.concatenate([base_feat, [0]]) if add_len else base_feat

    aac = np.array([seq.count(aa) / L for aa in AA20], dtype=float)

    thetas = []
    for pname, prop in props.items():
        theta = np.zeros(lam, dtype=float)
        for k in range(1, lam + 1):
            if L <= k: break
            diffs = [(prop[seq[i]] - prop[seq[i+k]]) ** 2 for i in range(L - k)]
            theta[k-1] = np.mean(diffs)
        thetas.append(theta)
    thetas = np.concatenate(thetas)

    denom = 1.0 + w * thetas.sum()
    feat = np.concatenate([aac, w * thetas]) / denom

    if add_len:
        feat = np.concatenate([feat, [L]])
    return feat

# ────────────────────────────────────────────────────────────
# 4. CTD features (set groups)
CTD_GROUPS = {
    "hydrophobicity": [set("RKEDQN"), set("GASTPHY"), set("CLVIMFW")],
    "normalized_vdw": [set("GASTPD"), set("NVEQIL"), set("MHKFRYW")],
    "polarizability": [set("GASDT"), set("CPNVEQIL"), set("KMHFRYW")],
    "secondary":      [set("NDEQST"), set("AILMV"), set("CFGHKPRYW")],
    "solvent_access": [set("ALFCGIVW"), set("RKQEND"), set("MPSTHY")],
    "polarity":       [set("LIFWCMVY"), set("PATGS"), set("HQRKNED")],
    "charge":         [set("KR"), set("ANCQGHILMFPSTWYV"), set("DE")]
}

def ctd_features(seq):
    seq = re.sub("[^ACDEFGHIKLMNPQRSTVWY]", "", seq.upper())
    L = len(seq)
    if L == 0:
        return np.zeros(len(CTD_GROUPS) * 21, dtype=float)

    all_feats = []
    for pname, groups in CTD_GROUPS.items():
        gseq = []
        for aa in seq:
            if aa in groups[0]: gseq.append(1)
            elif aa in groups[1]: gseq.append(2)
            elif aa in groups[2]: gseq.append(3)
            else: gseq.append(0)
        gseq = np.array(gseq)

        comp = [np.sum(gseq == g) / L for g in [1,2,3]]

        trans = []
        for g1 in [1,2,3]:
            for g2 in [1,2,3]:
                if g1 < g2:
                    cnt = np.sum((gseq[:-1]==g1)&(gseq[1:]==g2)) + \
                          np.sum((gseq[:-1]==g2)&(gseq[1:]==g1))
                    trans.append(cnt / (L-1))

        dist = []
        for g in [1,2,3]:
            idx = np.where(gseq == g)[0]
            if len(idx) == 0:
                dist.extend([0,0,0,0,0])
            else:
                dist.extend([
                    idx[0]/L,
                    idx[int(0.25*len(idx))]/L,
                    idx[int(0.50*len(idx))]/L,
                    idx[int(0.75*len(idx))]/L,
                    idx[-1]/L
                ])
        all_feats.extend(comp+trans+dist)

    return np.array(all_feats, dtype=float)  # 7*21=147 dims

# ────────────────────────────────────────────────────────────
# 5. Windowed AAC (3 segments × 20 dims = 60 dims)
def window_aac(seq, n_segments=3):
    seq = re.sub("[^ACDEFGHIKLMNPQRSTVWY]", "", seq.upper())
    L = len(seq)
    if L == 0:
        return np.zeros(n_segments * 20, dtype=float)

    seg_size = max(1, L // n_segments)
    feats = []
    for i in range(n_segments):
        start = i * seg_size
        end = L if i == n_segments-1 else (i+1)*seg_size
        sub = seq[start:end]
        if len(sub) == 0:
            feats.extend([0]*20)
        else:
            feats.extend([sub.count(aa)/len(sub) for aa in AA20])
    return np.array(feats, dtype=float)

# ────────────────────────────────────────────────────────────
# 6. Data loading
noenz_fasta = "./enz_or_not_dataset/uniprotkb_reviewed_true_AND_NOT_ec_2025_09_09.fasta"
enz_fasta   = "./enz_or_not_dataset/uniprotkb_reviewed_true_AND_ec_2025_09_09.fasta"

noenz_seqs = read_fasta(noenz_fasta)
enz_seqs   = read_fasta(enz_fasta)
print(f"Loaded {len(noenz_seqs)} non-enzyme sequences, {len(enz_seqs)} enzyme sequences")

# ────────────────────────────────────────────────────────────
# 7. Combined feature extraction
def extract_features(seq):
    pse = pse_aac_multi(seq, props=PROPERTIES)
    ctd = ctd_features(seq)
    win = window_aac(seq, n_segments=3)
    return np.concatenate([pse, ctd, win])

n_jobs = multiprocessing.cpu_count()
noenz_features = Parallel(n_jobs=n_jobs)(delayed(extract_features)(s) for s in noenz_seqs)
enz_features   = Parallel(n_jobs=n_jobs)(delayed(extract_features)(s) for s in enz_seqs)

X = np.vstack(noenz_features + enz_features)
y = np.array([0]*len(noenz_features) + [1]*len(enz_features))

print(f"Final feature dimensions: {X.shape[1]}")  # should be ≈ 258

# ────────────────────────────────────────────────────────────
# 8. Train / test split & model
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

clf = xgb.XGBClassifier(
    n_estimators=799,
    max_depth=9,
    learning_rate=0.11107207866102274,
    subsample=0.8205356041339127,
    colsample_bytree=0.7865743071602549,
    gamma=0.6350739428156533,
    min_child_weight=3,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=(len(y_train)-sum(y_train))/sum(y_train) if sum(y_train)>0 else 1,
    eval_metric="logloss",
)

try:
    clf.fit(X_train, y_train)
    y_pred  = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)

    print("\n===== XGBoost with Pseudo-AAC + Length + CTD + WindowAAC =====")
    print(classification_report(y_test, y_pred, target_names=["NoEnzyme", "Enzyme"]))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

    # Save model as enzyme_xgb_model.pkl
    joblib.dump(clf, "enzyme_xgb_model.pkl")
    print("Model saved as enzyme_xgb_model.pkl")

except Exception as e:
    print("Model training or prediction error:", e)